In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [2]:

data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_temporal_new.csv")

In [3]:
data.columns

Index(['baseline_updrs', 'subject', 'motor_updrs', 'test_time', 'updrs_lag1',
       'updrs_lag2', 'updrs_roll10', 'delta_lag1', 'updrs_from_baseline',
       'delta_motor_updrs', 'updrs_roll3', 'jitter', 'shimmer', 'hnr', 'rpde',
       'ppe', 'sex'],
      dtype='object')

In [4]:
data["updrs_delta"] = data["motor_updrs"] - data["updrs_lag1"]

In [5]:
features = [
    "updrs_lag1",
    "updrs_lag2",
    "updrs_roll10",
    "updrs_roll3",
    "delta_lag1",
    "updrs_from_baseline",
    "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex"
]

X = data[features]
target = "updrs_delta"

In [6]:
X = data[features]
y = data[target]
groups = data["subject"]

In [7]:
assert X.isna().sum().sum() == 0
assert len(X) == len(y)

In [8]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

In [9]:
ridge_pipeline.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge())])

In [10]:
import os
os.makedirs("../models", exist_ok=True)

In [11]:
import joblib

joblib.dump(
    ridge_pipeline,
    "../models/ridge_temporal_updrs.joblib"
)


['../models/ridge_temporal_updrs.joblib']

In [12]:
joblib.dump(
    features,
    "../models/temporal_features.joblib"
)


['../models/temporal_features.joblib']

In [13]:
metadata = {
    "target": target,
    "model": "Ridge Regression",
    "task": "Predict next-visit UPDRS delta",
    "features": features
}

joblib.dump(
    metadata,
    "../models/model_metadata.joblib"
)


['../models/model_metadata.joblib']

In [18]:
example_input = {
    "updrs_lag1": 1.2,
    "updrs_lag2": 0.8,
    "updrs_roll3": 1.0,
    "jitter": 0.006,
    "shimmer": 0.035,
    "hnr": 21.5,
    "rpde": 0.47,
    "ppe": 0.19,
    "sex": 1,
    'updrs_roll10': 1.0, 
    'delta_lag1': 1, 
    'updrs_from_baseline': 1.2
}


In [20]:
model = joblib.load("../models/ridge_temporal_updrs.joblib")
features = joblib.load("../models/temporal_features.joblib")

X_new = pd.DataFrame([example_input])[features]
predicted_delta = model.predict(X_new)[0]

In [22]:
def volatility_flag(std_delta, threshold=2.5):
    return "unstable" if std_delta > threshold else "stable"
